# Hướng dẫn huấn luyện mô hình 3D Reconstruction Baseline (ResNet50 + MLP)

Notebook này hướng dẫn bạn cách sử dụng bộ mã nguồn lõi để huấn luyện và đánh giá mô hình dự đoán Point Cloud 3D từ ảnh 2D.

## 1. Cài đặt môi trường

Đảm bảo bạn đã cài đặt các thư viện cần thiết trong `requirements.txt`.

```bash
pip install -r requirements.txt
```

Cấu trúc thư mục lõi:
- `src/models`: Chứa ResNet50 Encoder và MLP Decoder.
- `src/metrics`: Chứa các hàm tính loss (Chamfer Distance) và metrics (F-Score).
- `src/data`: Dataloader cho tập dữ liệu Pix3D.
- `src/training`: Pipeline huấn luyện.

In [ ]:
import torch
from pathlib import Path
import sys

# Thêm thư mục project vào path
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Project directory: {PROJECT_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Chuẩn bị dữ liệu

Sử dụng tập dữ liệu Pix3D. Bạn cần tải dữ liệu raw và chạy script tiền xử lý để tạo các file `.npy` (point cloud) và ảnh đã được crop.

```bash
python -m src.preprocessing.build_processed_dataset
```

## 3. Khởi tạo Mô hình

Mô hình sử dụng các dòng ResNet (18, 34, 50, 101, 152) làm Encoder để trích xuất đặc trưng ảnh và một Point Cloud Decoder (MLP hoặc RefineMLP) để dự đoán tọa độ các điểm 3D. 

**Lưu ý mới:** Hệ thống hiện đã có bảng tra `feature_dim` chuẩn. Ví dụ ResNet50 phải đi kèm với `feature_dim=2048` để tránh làm nhiễu đặc trưng pretrained.

In [ ]:
from src.models.object_reconstruction import build_object_reconstruction_model

device = "cuda" if torch.cuda.is_available() else "cpu"

model = build_object_reconstruction_model(
    encoder_name="resnet50",
    feature_dim=2048,           # Bắt buộc 2048 cho resnet50
    num_points=2048,
    pretrained=True,
    freeze_encoder=True,
    use_adapter=True,           # Kích hoạt PEFT Adapter
    adapter_bottleneck_dim=64   # Chiều bottleneck của adapter
).to(device)

print(f"Trainable parameters: {model.trainable_parameter_count():,}")

## 4. Huấn luyện (Training)

Bạn có thể chạy huấn luyện trực tiếp bằng `train.py` hoặc sử dụng `main_workflow.py` để tự động hóa cả bước tiền xử lý và training.

Ví dụ chạy lệnh từ terminal:
```bash
python train.py --categories chair --epochs 50 --batch-size 8 --output-dir results/my_experiment
```

Hoặc gọi hàm từ code:

In [ ]:
from src.training.training_pipeline import train_model

# Chú ý: Đây là ví dụ cấu hình tối giản
config = {
    "categories": ["chair"],
    "batch_size": 4,
    "epochs": 2,
    "lr": 1e-4,
    "output_dir": PROJECT_DIR / "results" / "smoke_test"
}

print("Bắt đầu huấn luyện thử nghiệm (smoke test)...")
# Lưu ý: Cần có dữ liệu trong data/processed để chạy thành công dòng dưới đây
# train_model(config)

## 5. Đánh giá (Evaluation)

Sau khi huấn luyện, bạn có thể đánh giá mô hình trên tập test sử dụng metrics Chamfer Distance và F-Score.

In [ ]:
from src.evaluation.evaluate_baseline import evaluate_model

# evaluate_model(checkpoint_path="results/my_experiment/outputs/checkpoints/best_model.pt")

## Kết luận

Đây là quy trình khép kín từ tiền xử lý đến huấn luyện và đánh giá. 
Để đạt kết quả tốt nhất, hãy thử huấn luyện với số lượng epoch lớn hơn (>100) và unfreeze encoder ở giai đoạn sau.